# 03 - Fine-tune YOLO on football data
**Question:** Does fine-tuning on labeled football images cut the extra boxes (spectators, false balls)
and separate the referee, while keeping the baseline's high recall?
**Method:** Train yolo11m on the Roboflow dataset (Ball / Player / Ref) at 1280 px, then rerun the
notebook-02 labeled-image test on the test split for both the baseline and the fine-tuned model.
**Outputs:** weights, training curves, metrics table, in outputs/03_finetune/.

In [8]:
import sys, shutil
from pathlib import Path
import cv2, numpy as np, pandas as pd, yaml
import torch
from ultralytics import YOLO

PROJECT = Path.cwd().parent
sys.path.append(str(PROJECT / "src"))
from football_eval import evaluate

DS = PROJECT / "data" / "detection"
OUT = PROJECT / "outputs" / "03_finetune"
OUT.mkdir(parents=True, exist_ok=True)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == 0 else "")

device: cpu 


In [9]:
cfg = yaml.safe_load(open(DS / "data.yaml"))
fixed = dict(path=DS.as_posix(), train="train/images", val="valid/images",
             test="test/images", nc=len(cfg["names"]), names=cfg["names"])
YAML = OUT / "data_fixed.yaml"
with open(YAML, "w") as f:
    yaml.safe_dump(fixed, f)
print(YAML.read_text())

names:
- Ball
- Player
- Ref
nc: 3
path: c:/Users/seree/OneDrive/Desktop/FalseNine/data/detection
test: test/images
train: train/images
val: valid/images



In [13]:
BEST = OUT / "train" / "weights" / "best.pt"
tuned = YOLO(str(BEST))
m = tuned.val(data=str(YAML), split="test", imgsz=1280, device=DEVICE,
              project=str(OUT), name="val_test", exist_ok=True)
print(f"mAP50 {m.box.map50:.3f}   mAP50-95 {m.box.map:.3f}")
for i, c in enumerate(m.box.ap_class_index):
    print(f"  {m.names[c]:7s} AP50 {m.box.ap50[i]:.3f}")

Ultralytics 8.4.174  Python-3.13.0 torch-2.14.1+cpu CPU (Intel Core i5-8365U 1.60GHz)
YOLO11m summary (fused): 125 layers, 20,032,345 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access  (ping: 0.20.0 ms, read: 453.033.1 MB/s, size: 122.4 KB)
val: Scanning C:\Users\seree\OneDrive\Desktop\FalseNine\data\detection\test\labels... 19 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 19/19 919.9it/s 0.0s
val: New cache created: C:\Users\seree\OneDrive\Desktop\FalseNine\data\detection\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 15.3s/it 30.5s1:21
                   all         19        255      0.968      0.886      0.926      0.607
                  Ball         15         17          1      0.697       0.81      0.394
                Player         19        225      0.967       0.96      0.972      0.678
                   Ref         10         13      0.936          1      0.995      0.748
S

In [15]:
base = YOLO("yolo11m.pt")

def predict_with(model, classes=None):
    def _p(img):
        r = model.predict(img, imgsz=1280, conf=0.10, classes=classes,
                          device=DEVICE, verbose=False)[0]
        return r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.xyxy.cpu().numpy()
    return _p

before = evaluate(predict_with(base, [0, 32]), DS, ["test"], pred_ball={32}, pred_person={0})
after = evaluate(predict_with(tuned), DS, ["test"], pred_ball={0}, pred_person={1, 2})

cmp = pd.DataFrame([before, after], index=["baseline (pretrained)", "fine-tuned"])
cmp.to_csv(OUT / "comparison_test_split.csv")
cmp

,ball_found,ball_total,ball_extra,person_found,person_total,person_extra
baseline (pretrained),13,17,8,237,238,61
fine-tuned,13,17,2,233,238,2


In [6]:
(PROJECT / "models").mkdir(exist_ok=True)
shutil.copy(BEST, PROJECT / "models" / "falsenine_det.pt")
print("saved", PROJECT / "models" / "falsenine_det.pt")

saved c:\Users\seree\OneDrive\Desktop\FalseNine\models\falsenine_det.pt


## Findings (test split: 19 images, 255 labeled objects; yolo11m, 1280 px)
- Official scores: mAP50 0.926, mAP50-95 0.607. AP50 by class: Ball 0.810, Player 0.972, Ref 0.995.
- Same images, pretrained vs fine-tuned (found / extra boxes):
  ball 13/17 and 8 extra -> 13/17 and 2 extra; people 237/238 and 61 extra -> 233/238 and 2 extra.
- 46 missing goalkeeper/player boxes were added to the dataset labels (class Player) before this run.
- Fine-tuning removed nearly all false person boxes (61 -> 2) at a small recall cost (237 -> 233 of 238)
  and added a separate referee class.
- Ball remains the weakest class: found 13 of 17 in both models; mAP50-95 0.39 (tiny object, loose boxes).
- Retraining on slightly different labels moved scores by about 0.01, so differences that small are noise.
- Limits: tiny test set (17 balls); train and test frames come from the same match, so scores are
  optimistic. A cross-match check is planned in notebook 04.
- Speed: about 1.6 s per image on CPU at 1280 px.